In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}
df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)


Parte 1: Inicialização e Marcadores Básicos


In [ ]:
# 1. Mapa base centralizado na coordenada média
lat_media = df_mapa['latitude'].mean()
lon_media = df_mapa['longitude'].mean()

mapa_basico = folium.Map(
    location=[lat_media, lon_media],
    zoom_start=12,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri'
)

In [ ]:
# 2. Marcadores simples para as 5 primeiras linhas
for idx, row in df_mapa.head(5).iterrows():
    popup_texto = (
        f"Tipo: {row['tipo']}<br>"
        f"Valor: R$ {row['valor_venda']:,.2f}"
    )
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_texto, max_width=250),
        tooltip=f"Imóvel #{row['id_imovel']}"
    ).add_to(mapa_basico)

mapa_basico

Parte 2: Customização Visual com Marcadores Circulares

In [ ]:
# 3. Novo mapa base
mapa_circulos = folium.Map(
    location=[lat_media, lon_media],
    zoom_start=12,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri'
)

In [ ]:
# 4. CircleMarker para todos os imóveis
cores_cidade = {
    'Nova Iguaçu': 'blue',
    'Queimados': 'orange'
}

for idx, row in df_mapa.iterrows():
    cor = cores_cidade.get(row['cidade'], 'gray')
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.8,
        weight=1,
        tooltip="Clique para detalhes"
    ).add_to(mapa_circulos)

mapa_circulos

Parte 3: Agrupamento Inteligente (Clustering)

In [ ]:
# 5. Terceiro mapa base + MarkerCluster
mapa_cluster = folium.Map(
    location=[lat_media, lon_media],
    zoom_start=12,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri'
)

marker_cluster = MarkerCluster(name='Imóveis').add_to(mapa_cluster)

In [ ]:
# 6. Ícones customizados por tipo, adicionados ao cluster
cores_tipo = {
    'Casa': 'green',
    'Apartamento': 'blue',
    'Terreno': 'gray'
}

for idx, row in df_mapa.iterrows():
    cor = cores_tipo.get(row['tipo'], 'gray')
    popup_texto = (
        f"<b>Imóvel #{row['id_imovel']}</b><br>"
        f"Cidade: {row['cidade']}<br>"
        f"Tipo: {row['tipo']}<br>"
        f"Valor: R$ {row['valor_venda']:,.2f}"
    )
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=folium.Popup(popup_texto, max_width=250),
        tooltip=row['tipo'],
        icon=folium.Icon(color=cor, icon='info-sign')
    ).add_to(marker_cluster)

In [ ]:
# 7. Salva o mapa final
mapa_cluster.save('mapa_imoveis_baixada.html')
mapa_cluster